In [1]:
import duckdb
con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")

url = "https://d37ci6vzurychx.cloudfront.net/trip-data/fhvhv_tripdata_2025-01.parquet"

print(con.sql(f"DESCRIBE SELECT * FROM '{url}'"))

print(con.sql(f"""
    SELECT COUNT(*) AS trips,
           SUM(cbd_congestion_fee) AS toll_revenue
    FROM '{url}'
"""))

┌──────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│     column_name      │ column_type │  null   │   key   │ default │  extra  │
│       varchar        │   varchar   │ varchar │ varchar │ varchar │ varchar │
├──────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ hvfhs_license_num    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ dispatching_base_num │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ originating_base_num │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ request_datetime     │ TIMESTAMP   │ YES     │ NULL    │ NULL    │ NULL    │
│ on_scene_datetime    │ TIMESTAMP   │ YES     │ NULL    │ NULL    │ NULL    │
│ pickup_datetime      │ TIMESTAMP   │ YES     │ NULL    │ NULL    │ NULL    │
│ dropoff_datetime     │ TIMESTAMP   │ YES     │ NULL    │ NULL    │ NULL    │
│ PULocationID         │ INTEGER     │ YES     │ NULL    │ NULL    │ NULL    │
│ DOLocationID         │ INTEGER     │ YES     │ NUL

In [2]:
import pandas as pd
pd.set_option("display.max_rows", 100)

zones_url = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"

cbd_check = con.sql(f"""
    SELECT t.PULocationID AS zone_id,
           z.Borough, z.Zone,
           COUNT(*) AS trips,
           ROUND(AVG(CASE WHEN t.cbd_congestion_fee > 0 THEN 1 ELSE 0 END), 3) AS pct_tolled
    FROM '{url}' t
    JOIN read_csv_auto('{zones_url}') z ON t.PULocationID = z.LocationID
    WHERE t.PULocationID = t.DOLocationID
    GROUP BY ALL
    HAVING COUNT(*) > 100
    ORDER BY pct_tolled DESC
""").df()

cbd_check.head(80)

,zone_id,Borough,Zone,trips,pct_tolled
0,125,Manhattan,Hudson Sq,615,0.941
1,231,Manhattan,TriBeCa/Civic Center,10957,0.919
2,113,Manhattan,Greenwich Village North,1717,0.918
3,162,Manhattan,Midtown East,1957,0.911
4,234,Manhattan,Union Sq,3848,0.910
5,87,Manhattan,Financial District North,1217,0.908
6,246,Manhattan,West Chelsea/Hudson Yards,5983,0.903
7,161,Manhattan,Midtown Center,4081,0.902
8,158,Manhattan,Meatpacking/West Village West,1962,0.900
9,45,Manhattan,Chinatown,500,0.894


In [3]:
inside = cbd_check[cbd_check.pct_tolled >= 0.8]
middle = cbd_check[(cbd_check.pct_tolled > 0.05) & (cbd_check.pct_tolled < 0.8)]

print("Zones clearly inside:", len(inside))
print(inside[["zone_id", "Zone"]].to_string())
print("\nBorderline zones:")
print(middle[["zone_id", "Borough", "Zone", "pct_tolled"]].to_string())

Zones clearly inside: 37
    zone_id                            Zone
0       125                       Hudson Sq
1       231            TriBeCa/Civic Center
2       113         Greenwich Village North
3       162                    Midtown East
4       234                        Union Sq
5        87        Financial District North
6       246       West Chelsea/Hudson Yards
7       161                  Midtown Center
8       158   Meatpacking/West Village West
9        45                       Chinatown
10      211                            SoHo
11      170                     Murray Hill
12       13               Battery Park City
13       90                        Flatiron
14      114         Greenwich Village South
15      163                   Midtown North
16      144             Little Italy/NoLiTa
17      107                        Gramercy
18       68                    East Chelsea
19      229   Sutton Place/Turtle Bay North
20      137                        Kips Bay
21     

In [4]:
zones_code = '''# NYC Congestion Relief Zone - TLC zone IDs
# Derived from Jan 2025 HVFHV data: share of same-zone trips charged cbd_congestion_fee

CBD_ZONES = [
    4, 12, 13, 45, 48, 50, 68, 79, 87, 88, 90, 100, 107, 113, 114, 125,
    137, 144, 148, 158, 161, 162, 163, 164, 170, 186, 209, 211, 224, 229,
    230, 231, 232, 233, 234, 246, 249, 261,
]

# Zones straddling 60th St - partly tolled, excluded from DiD
BOUNDARY_ZONES = [43, 140, 141, 142, 143, 237]
'''

with open("../src/zones.py", "w") as f:
    f.write(zones_code)

print("Saved src/zones.py")

FileNotFoundError: [Errno 2] No such file or directory: '../src/zones.py'

In [5]:
import os

repo = r"C:\Users\lalit\Documents\nyc-congestion-toll-impact"
for folder in ["data", "notebooks", "src", "outputs"]:
    os.makedirs(os.path.join(repo, folder), exist_ok=True)

with open(os.path.join(repo, "src", "zones.py"), "w") as f:
    f.write(zones_code)

print("Notebook is running in:", os.getcwd())
print("Saved src/zones.py")

Notebook is running in: C:\Users\lalit\Documents\nyc-congestion-toll-impact
Saved src/zones.py
